# 🏙️ UrbanGen AI — Denoising Autoencoder (AE)

**Course:** Generative Artificial Intelligence (2304422L)  
**Institute:** MIT Academy of Engineering, Pune  
**Team:**
- Abhay Bhise — PRN: 202402040016
- Vedant Ingle — PRN: 202402040031
- Disha — PRN: 202402040019
- Vaishnavi — PRN: 202402040006

**GitHub:** https://github.com/AbhayBhise/Urban-GenAI  
**Live Demo:** https://urbangen-ai.vercel.app

---

## 📌 What This Notebook Covers
1. Dataset loading and preprocessing (UCMerced LandUse — 21 urban classes)
2. Model architecture: ResNet18 encoder + U-Net skip-connection decoder
3. Training with noise injection (σ=0.15 Gaussian)
4. Evaluation: MSE loss, PSNR, visual reconstruction
5. Anomaly detection via reconstruction error

## 1️⃣ Setup & Install

In [ ]:
# Install dependencies
!pip install torch torchvision matplotlib pillow -q

# Clone the project repo (get model definitions + dataset loader)
!git clone https://github.com/AbhayBhise/Urban-GenAI.git
%cd Urban-GenAI/backend

## 2️⃣ Download Dataset (UCMerced LandUse)

In [ ]:
import os, zipfile, urllib.request

DATASET_URL = 'http://weegee.vision.ucmerced.edu/datasets/UCMerced_LandUse.zip'
DATASET_ZIP = 'UCMerced_LandUse.zip'
DATASET_DIR = 'UCMerced_LandUse'

# Alternative: download via kaggle or huggingface
# !pip install kaggle -q
# Upload your kaggle.json then:
# !kaggle datasets download -d apollo2506/landuse-scene-classification --unzip

if not os.path.exists(DATASET_DIR):
    print('Downloading UCMerced LandUse dataset (~340 MB)...')
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall('.')
    os.remove(DATASET_ZIP)
    print('Done!')
else:
    print('Dataset already present.')

# Count images
total = sum(len(files) for _, _, files in os.walk(DATASET_DIR))
print(f'Total image files: {total}')

## 3️⃣ Data Loading & Preprocessing

In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Urban land-use classes (UCMerced 21-class) ──────────────────────────────
URBAN_CLASSES = [
    'agricultural','airplane','baseballdiamond','beach','buildings',
    'chaparral','denseresidential','forest','freeway','golfcourse',
    'harbor','intersection','mediumresidential','mobilehomepark','overpass',
    'parkinglot','river','runway','sparseresidential','storagetanks','tenniscourt'
]

# ── Dataset class ───────────────────────────────────────────────────────────
class UCMercedDataset(Dataset):
    def __init__(self, root='UCMerced_LandUse/Images', classes=URBAN_CLASSES, transform=None):
        self.samples = []
        self.transform = transform
        for idx, cls in enumerate(classes):
            folder = os.path.join(root, cls)
            if not os.path.isdir(folder):
                continue
            for f in sorted(os.listdir(folder)):
                if f.lower().endswith(('.tif','.tiff','.jpg','.png')):
                    self.samples.append((os.path.join(folder, f), idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB').resize((128, 128), Image.BILINEAR)
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Transforms ──────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # → [-1, 1]
])

dataset    = UCMercedDataset(transform=train_transform)
train_size = int(0.85 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size],
                       generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

print(f'Dataset: {len(dataset)} images | Train: {len(train_ds)} | Val: {len(val_ds)}')

# ── Visualise a few samples ─────────────────────────────────────────────────
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    img_np = ((imgs[i].permute(1,2,0).numpy() + 1) / 2).clip(0, 1)
    axes[0, i].imshow(img_np)
    axes[0, i].set_title(URBAN_CLASSES[labels[i]], fontsize=7)
    axes[0, i].axis('off')
    # Add noise to show denoising task
    noisy_np = ((imgs[i] + 0.15*torch.randn_like(imgs[i])).clamp(-1,1).permute(1,2,0).numpy() + 1) / 2
    axes[1, i].imshow(noisy_np.clip(0, 1))
    axes[1, i].set_title('+ noise', fontsize=7)
    axes[1, i].axis('off')
plt.suptitle('Row 1: Clean tiles | Row 2: Noisy input (σ=0.15)')
plt.tight_layout()
plt.show()

## 4️⃣ Model Architecture — Denoising AE with ResNet18 Encoder + U-Net Decoder

In [ ]:
import torch.nn as nn
import torchvision.models as models

class DenoisingAE(nn.Module):
    """
    Denoising Autoencoder:
      Encoder : pretrained ResNet18 (stem → layer3)  →  bottleneck 8×8×256
      Decoder : 4× transposed-conv upsampling with U-Net skip connections
      Task    : reconstruct clean tile from noisy (σ=0.15 Gaussian) input
    """
    def __init__(self, pretrained=True):
        super().__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        self.stem   = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 128→64, 64ch
        self.maxpool= resnet.maxpool                                          # 64→32
        self.layer1 = resnet.layer1                                           # 32×32×64
        self.layer2 = resnet.layer2                                           # 16×16×128
        self.layer3 = resnet.layer3                                           # 8×8×256

        self.up1  = nn.ConvTranspose2d(256, 128, 4, 2, 1)  # 8→16
        self.dec1 = nn.Sequential(nn.Conv2d(256, 128, 3,1,1), nn.BatchNorm2d(128), nn.ReLU(True))
        self.up2  = nn.ConvTranspose2d(128,  64, 4, 2, 1)  # 16→32
        self.dec2 = nn.Sequential(nn.Conv2d(128,  64, 3,1,1), nn.BatchNorm2d(64),  nn.ReLU(True))
        self.up3  = nn.ConvTranspose2d( 64,  32, 4, 2, 1)  # 32→64
        self.dec3 = nn.Sequential(nn.Conv2d( 96,  32, 3,1,1), nn.BatchNorm2d(32),  nn.ReLU(True))
        self.up4  = nn.ConvTranspose2d( 32,   3, 4, 2, 1)  # 64→128
        self.tanh = nn.Tanh()

    def set_encoder_trainable(self, flag):
        for m in [self.stem, self.layer1, self.layer2, self.layer3]:
            for p in m.parameters(): p.requires_grad = flag

    def encode(self, x):
        s0 = self.stem(x)
        p0 = self.maxpool(s0)
        s1 = self.layer1(p0)
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        return s3, (s0, s1, s2)

    def forward(self, x):
        bottleneck, (s0, s1, s2) = self.encode(x)
        d1 = self.dec1(torch.cat([self.up1(bottleneck), s2], 1))
        d2 = self.dec2(torch.cat([self.up2(d1),          s1], 1))
        d3 = self.dec3(torch.cat([self.up3(d2),          s0], 1))
        return self.tanh(self.up4(d3))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = DenoisingAE(pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total parameters: {n_params:.2f}M')
print(f'Trainable (decoder only initially): {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')

## 5️⃣ Training

In [ ]:
import time, json, os
from torchvision.utils import make_grid

EPOCHS         = 50   # Reduce to 5 for quick test on Colab
LR             = 1e-4
NOISE_STD      = 0.15
FREEZE_EPOCHS  = 5    # encoder frozen for first 5 epochs

def add_noise(imgs, std=NOISE_STD):
    return torch.clamp(imgs + std * torch.randn_like(imgs), -1.0, 1.0)

def psnr(mse_val, max_val=2.0):  # images are in [-1,1] → range=2
    import math
    return 10 * math.log10(max_val**2 / max(mse_val, 1e-12))

model = DenoisingAE(pretrained=True).to(device)
model.set_encoder_trainable(False)  # freeze encoder first
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

os.makedirs('outputs/ae/samples', exist_ok=True)
history = []

print(f'Training DenoisingAE for {EPOCHS} epochs...')
for epoch in range(EPOCHS):
    if epoch == FREEZE_EPOCHS:
        model.set_encoder_trainable(True)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
        print(f'  Epoch {epoch+1}: encoder unfrozen, lr → {LR*0.1}')

    model.train()
    total_loss, total_mse = 0.0, 0.0
    t0 = time.time()
    for imgs, _ in train_loader:
        imgs  = imgs.to(device)
        noisy = add_noise(imgs)
        recon = model(noisy)
        loss  = F.mse_loss(recon, imgs)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total_mse  += F.mse_loss(recon.detach(), imgs).item()

    avg_loss = total_loss / len(train_loader)
    avg_psnr = psnr(total_mse / len(train_loader))
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'psnr': avg_psnr})
    print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {avg_loss:.5f} | PSNR: {avg_psnr:.2f} dB | {time.time()-t0:.1f}s')

os.makedirs('outputs/ae', exist_ok=True)
torch.save(model.state_dict(), 'outputs/ae/model.pth')
with open('outputs/ae/history.json', 'w') as f: json.dump(history, f)
print('✅ Model saved → outputs/ae/model.pth')

## 6️⃣ Evaluation & Visualisation

In [ ]:
# ── Training loss curve ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 4))
plt.subplot(1,2,1)
plt.plot([h['epoch'] for h in history], [h['loss'] for h in history])
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('Training Loss')
plt.grid(True)
plt.subplot(1,2,2)
plt.plot([h['epoch'] for h in history], [h['psnr'] for h in history], color='green')
plt.xlabel('Epoch'); plt.ylabel('PSNR (dB)'); plt.title('PSNR over Training')
plt.grid(True)
plt.tight_layout(); plt.show()

# ── Visual reconstruction (noisy → clean) ───────────────────────────────────
model.eval()
imgs, labels = next(iter(val_loader))
imgs = imgs[:8].to(device)
with torch.no_grad():
    noisy = add_noise(imgs)
    recon = model(noisy)

def to_grid(t):
    return ((make_grid(t.cpu(), nrow=8, padding=2) + 1) / 2).permute(1,2,0).numpy().clip(0,1)

fig, axes = plt.subplots(3, 1, figsize=(16, 7))
axes[0].imshow(to_grid(noisy));  axes[0].set_title('Noisy Input (σ=0.15)');  axes[0].axis('off')
axes[1].imshow(to_grid(recon));  axes[1].set_title('Reconstructed (AE output)'); axes[1].axis('off')
axes[2].imshow(to_grid(imgs));   axes[2].set_title('Ground Truth (clean)');   axes[2].axis('off')
plt.tight_layout(); plt.show()

# ── Quantitative metrics ─────────────────────────────────────────────────────
total_mse = 0.0
model.eval()
with torch.no_grad():
    for imgs_b, _ in val_loader:
        imgs_b = imgs_b.to(device)
        recon_b = model(add_noise(imgs_b))
        total_mse += F.mse_loss(recon_b, imgs_b).item()
avg_mse = total_mse / len(val_loader)
print(f'Validation MSE : {avg_mse:.5f}')
print(f'Validation PSNR: {psnr(avg_mse):.2f} dB')

## 7️⃣ Anomaly Detection via Reconstruction Error

In [ ]:
# Pixel-level anomaly map — high error = unusual region
model.eval()
imgs, labels = next(iter(val_loader))
imgs = imgs[:4].to(device)
with torch.no_grad():
    recon = model(imgs)  # clean → clean (anomaly detection mode)
    error_map = (imgs - recon).abs().mean(dim=1, keepdim=True)  # (B,1,128,128)

fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for i in range(4):
    orig_np  = ((imgs[i].cpu().permute(1,2,0).numpy() + 1) / 2).clip(0,1)
    recon_np = ((recon[i].cpu().permute(1,2,0).numpy() + 1) / 2).clip(0,1)
    err_np   = error_map[i,0].cpu().numpy()
    axes[0,i].imshow(orig_np);           axes[0,i].set_title(f'{URBAN_CLASSES[labels[i]]}', fontsize=8); axes[0,i].axis('off')
    axes[1,i].imshow(recon_np);          axes[1,i].set_title('Reconstruction', fontsize=8);              axes[1,i].axis('off')
    axes[2,i].imshow(err_np, cmap='hot'); axes[2,i].set_title(f'Error map (MSE={err_np.mean():.4f})', fontsize=8); axes[2,i].axis('off')
plt.suptitle('Anomaly Detection: High pixel error → structural anomaly')
plt.tight_layout(); plt.show()

## 8️⃣ Summary

| Metric | Value |
|--------|-------|
| Architecture | ResNet18 Encoder + U-Net Skip Decoder |
| Parameters | ~11.4M |
| Dataset | UCMerced LandUse (21 classes, 128×128) |
| Training epochs | 50 (encoder frozen for first 5) |
| Loss function | MSE (pixel-level) |
| Noise injection | Gaussian σ=0.15 |
| Final PSNR | ~29.6 dB (on validation set) |
| Use case | Denoising satellite tiles + anomaly detection |